In [ ]:
# Instalar bibliotecas necessárias
!pip install -q pandas matplotlib wordcloud kaggle

##### OBTENÇÃO DE DADOS ATUALIZAODS VIA API BANCO CENTRAL

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta

url = "https://olinda.bcb.gov.br/olinda/servico/MPV_DadosAbertos/versao/v1/odata/MeiosdePagamentosMensalDA?$top=1000&$format=json&$select=AnoMes,quantidadePix,valorPix,quantidadeTED,valorTED,quantidadeTEC,valorTEC,quantidadeCheque,valorCheque,quantidadeBoleto,valorBoleto,quantidadeDOC,valorDOC"

response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    payment_methods = pd.DataFrame(data['value'])

	# Translate column names from original in Brazilian Portuguese to English
    column_mapping = {
            "AnoMes": "YearMonth",
            "quantidadePix": "quantityPix",
            "valorPix": "valuePix",
            "quantidadeTED": "quantityTED",
            "valorTED": "valueTED",
            "quantidadeTEC": "quantityTEC",
            "valorTEC": "valueTEC",
            "quantidadeCheque": "quantityBankCheck",
            "valorCheque": "valueBankCheck",
            "quantidadeBoleto": "quantityBrazilianBoletoPayment",
            "valorBoleto": "valueBrazilianBoletoPayment",
            "quantidadeDOC": "quantityDOC",
            "valorDOC": "valueDOC"
    }

    # Rename DataFrame columns
    payment_methods.rename(columns=column_mapping, inplace=True)

    # Save data in CSV file.
    payment_methods.to_csv('brazilian_payment_methods.csv', index=False)

else:
    print(f"Request Error: {response.status_code}")

##### OBTENÇÃO DE DADOS VIA KAGGLE

In [ ]:
# Fazer upload do arquivo kaggle.json
from google.colab import files
files.upload()

# Mover o arquivo para o local correto
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Baixar os dados do Kaggle
!kaggle datasets download -d clovisdalmolinvieira/brazilian-payment-methods
!unzip brazilian-payment-methods.zip

##### ANÁLISE DE DADOS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud, ImageColorGenerator
from PIL import Image

In [ ]:
# Carregar dados de métodos de pagamento
dados = pd.read_csv('brazilian_payment_methods.csv')

# Visualizar as primeiras linhas do DataFrame
dados.head()

In [ ]:
# Derretendo o DataFrame para transformar colunas em linhas
df_quantity = pd.melt(dados, id_vars=['YearMonth'],
                      value_vars=['quantityPix', 'quantityTED', 'quantityTEC', 'quantityBankCheck', 'quantityBrazilianBoletoPayment', 'quantityDOC'],
                      var_name='PaymentMethod', value_name='Quantity')

df_value = pd.melt(dados, id_vars=['YearMonth'],
                   value_vars=['valuePix', 'valueTED', 'valueTEC', 'valueBankCheck', 'valueBrazilianBoletoPayment', 'valueDOC'],
                   var_name='PaymentMethod', value_name='Value')

# Substituindo nomes das colunas para apenas o método de pagamento
df_quantity['PaymentMethod'] = df_quantity['PaymentMethod'].str.replace('quantity', '')
df_value['PaymentMethod'] = df_value['PaymentMethod'].str.replace('value', '')

# Combinando os DataFrames de quantidade e valor
df_restruturado = pd.merge(df_quantity, df_value, on=['YearMonth', 'PaymentMethod'])

# Visualizar as primeiras linhas do DataFrame
df_restruturado.head()

In [ ]:
df_restruturado['PaymentMethod'] = df_restruturado['PaymentMethod'].replace({
  'Pix': 'PIX',
  'TED': 'TED',
  'TEC': 'TEC',
  'DOC': 'DOC',
  'BankCheck': 'Cheque',
  'BrazilianBoletoPayment': 'Boleto',
})

In [ ]:
# Analisar a frequência dos métodos de pagamento
metodos_pagamento = df_restruturado.groupby('PaymentMethod')['Quantity'].sum().reset_index()
metodos_pagamento.sort_values(by='Quantity', ascending=False, inplace=True)

# Exibir os métodos de pagamento mais populares
print(metodos_pagamento.head(10))

In [ ]:
# Carregar a imagem da bandeira do Brasil
from google.colab import files
files.upload()
mask = np.array(Image.open('Brasil.png'))
image_colors = ImageColorGenerator(mask)

In [ ]:
# Gerar uma nuvem de palavras para os métodos de pagamento
wordcloud = WordCloud(width=800, height=400, background_color='black', mask=mask).generate(' '.join(metodos_pagamento['PaymentMethod'].unique()))

# Visualizar a nuvem de palavras
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud.recolor(color_func=image_colors), interpolation='bilinear')
plt.axis('off')
plt.show()    [1]
check


## 1️⃣ Preparar os dados de tempo

In [ ]:
# Converter YearMonth para formato de data
df_restruturado['YearMonth'] = pd.to_datetime(df_restruturado['YearMonth'], format='%Y%m')

# Ordenar por data
df_restruturado = df_restruturado.sort_values('YearMonth')

df_restruturado.head()

## 2️⃣ Evolução da quantidade de pagamentos (responde mudança de método)

In [ ]:
# Criar tabela pivot para análise temporal
pivot_quantidade = df_restruturado.pivot_table(
    index='YearMonth',
    columns='PaymentMethod',
    values='Quantity',
    aggfunc='sum'
)

pivot_quantidade.plot(figsize=(12,6))
plt.title("Evolução dos meios de pagamento no Brasil")
plt.ylabel("Quantidade de transações")
plt.xlabel("Ano")
plt.show()

## 3️⃣ Descobrir automaticamente qual era o método mais usado em cada período

In [ ]:
# Identificar o método mais utilizado por mês
mais_usado_por_mes = pivot_quantidade.idxmax(axis=1)

mais_usado_por_mes.head(20)

In [ ]:
mudancas = mais_usado_por_mes[mais_usado_por_mes != mais_usado_por_mes.shift()]
mudancas

## 4️⃣ Comparar quantidade vs valor financeiro

In [ ]:
pivot_valor = df_restruturado.pivot_table(
    index='YearMonth',
    columns='PaymentMethod',
    values='Value',
    aggfunc='sum'
)

pivot_valor.plot(figsize=(12,6))
plt.title("Valor financeiro movimentado por meio de pagamento")
plt.ylabel("Valor total")
plt.xlabel("Ano")
plt.show()

## 5️⃣ Análise de correlação

In [ ]:
# correlação entre meios de pagamento
correlacao = pivot_quantidade.corr()

correlacao

In [ ]:
import seaborn as sns

plt.figure(figsize=(8,6))
sns.heatmap(correlacao, annot=True, cmap="coolwarm")
plt.title("Correlação entre meios de pagamento")
plt.show()